# Fixed-k5 vs Silhouette Evaluation
This notebook loads the filtered evaluation results and compares mean/max AUC and MAD metrics between fixed-k5 and silhouette methods.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 12

RESULTS_PATH = '../results/cosy-evaluation_GPT-explain_fixed-silhouette.csv'
df = pd.read_csv(RESULTS_PATH)
print(f'Loaded {len(df)} rows from {RESULTS_PATH}')

df_fixed = df[df['method'] == 'fixed-k5'].copy()
df_silhouette = df[df['method'] == 'silhouette'].copy()
print(f'Fixed rows: {len(df_fixed)}')
print(f'Silhouette rows: {len(df_silhouette)}')

In [ ]:
def process_groups(groups):
    all_max_auc = []
    all_max_mad = []
    for (_, _), group in groups:
        all_max_auc.append(group['AUC'].max())
        all_max_mad.append(group['MAD'].max())
    return np.array(all_max_auc), np.array(all_max_mad)

def get_confidence_interval(data, confidence=0.95):
    a = np.array(data)
    if len(a) <= 1:
        return np.mean(a), np.mean(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2.0, len(a) - 1)
    return m - h, m + h

def summarize(df, metric='mean'):
    if metric == 'mean':
        auc_mean = df['AUC'].mean()
        auc_lower, auc_upper = get_confidence_interval(df['AUC'])
        mad_mean = df['MAD'].mean()
        mad_std = df['MAD'].std()
    elif metric == 'max':
        groups = df.groupby(['layer','unit'])
        auc_vals, mad_vals = process_groups(groups)
        auc_mean = auc_vals.mean()
        auc_lower, auc_upper = get_confidence_interval(auc_vals)
        mad_mean = mad_vals.mean()
        mad_std = mad_vals.std()
    return auc_mean, auc_lower, auc_upper, mad_mean, mad_std

In [ ]:
rows = []
for label, subset in [('fixed-k5', df_fixed), ('silhouette', df_silhouette)]:
    mean_auc, mean_low, mean_up, mean_mad, std_mad = summarize(subset, 'mean')
    max_auc, max_low, max_up, max_mad, std_max_mad = summarize(subset, 'max')
    rows.append({
        'method': label,
        'mean_auc (CI)': f"{mean_auc:.2f} ({mean_low:.2f}-{mean_up:.2f})",
        'mean_mad (±std)': f"{mean_mad:.2f} ± {std_mad:.2f}",
        'max_auc (CI)': f"{max_auc:.2f} ({max_low:.2f}-{max_up:.2f})",
        'max_mad (±std)': f"{max_mad:.2f} ± {std_max_mad:.2f}",
        'num_rows': len(subset)
    })
comparison_df = pd.DataFrame(rows)
comparison_df

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(comparison_df['method'], [df_fixed['AUC'].mean(), df_silhouette['AUC'].mean()])
plt.ylabel('Mean AUC')
plt.title('Mean AUC by Method')
plt.show()

plt.figure(figsize=(8,4))
plt.bar(comparison_df['method'], [df_fixed['MAD'].mean(), df_silhouette['MAD'].mean()])
plt.ylabel('Mean MAD')
plt.title('Mean MAD by Method')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df_fixed['AUC'], bins=30, alpha=0.5, label='fixed-k5')
plt.hist(df_silhouette['AUC'], bins=30, alpha=0.5, label='silhouette')
plt.legend()
plt.xlabel('AUC')
plt.ylabel('Count')
plt.title('AUC Distribution by Method')
plt.show()

In [ ]:
grouped = df.groupby(['layer','unit','method'])['AUC'].mean().reset_index()
plt.figure(figsize=(10,5))
for method, marker in [('fixed-k5','o'),('silhouette','s')]:
    subset = grouped[grouped['method']==method]
    plt.scatter(subset['layer'] + subset['unit']/10000, subset['AUC'], label=method, alpha=0.6, marker=marker)
plt.legend()
plt.xlabel('Layer + unit/1e4')
plt.ylabel('Mean AUC per neuron')
plt.title('Per-neuron AUC Comparison')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
colors = {'fixed-k5':'tab:blue','silhouette':'tab:orange'}
for method in ['fixed-k5','silhouette']:
    subset = df[df['method']==method]
    plt.scatter(subset['MAD'], subset['AUC'], alpha=0.4, label=method, c=colors[method])
plt.legend()
plt.xlabel('MAD')
plt.ylabel('AUC')
plt.title('AUC vs MAD by Method')
plt.show()

## Extended Analysis: K-Value Groups
### Investigating the bimodal k-distribution and its impact on performance

In [ ]:
import os
import re
from scipy.stats import gaussian_kde
import seaborn as sns

# Extract k-values for each neuron from description files
desc_dir = '../descriptions/gemini-2-5-flash/gpt2-xl'
neuron_k = {}

for fname in os.listdir(desc_dir):
    if 'silhouette' in fname and fname.endswith('.csv'):
        parts = fname.replace('gpt2-xl_', '').split('_')
        layer = int(parts[0].replace('layer-', ''))
        unit = int(parts[1].replace('unit-', ''))
        
        df_desc = pd.read_csv(os.path.join(desc_dir, fname))
        k = len(df_desc)
        neuron_k[(layer, unit)] = k

print(f'Extracted k-values for {len(neuron_k)} neurons')
print(f'\nK-value distribution:')
from collections import Counter
k_counts = Counter(neuron_k.values())
for k in sorted(k_counts.keys()):
    print(f'  k={k}: {k_counts[k]} neurons ({k_counts[k]/len(neuron_k)*100:.1f}%)')

In [ ]:
# Add k-values to dataframe
df['k'] = df.apply(lambda row: neuron_k.get((int(row['layer']), int(row['unit'])), None), axis=1)
df['k_group'] = df['k'].apply(lambda x: 'low_k (k≤5)' if pd.notna(x) and x <= 5 else 'high_k (k>5)' if pd.notna(x) else None)

print('\nK-group distribution for silhouette:')
print(df[df['method']=='silhouette']['k_group'].value_counts())

### Performance Comparison: Fixed-k5 vs Silhouette by K-Group

In [ ]:
# Compare performance for low-k vs high-k neurons
results = []

for k_group in ['low_k (k≤5)', 'high_k (k>5)']:
    sil_subset = df[(df['method'] == 'silhouette') & (df['k_group'] == k_group)]
    neurons_in_group = sil_subset.groupby(['layer', 'unit']).size().index
    
    fixed_subset = df[df['method'] == 'fixed-k5']
    fixed_subset = fixed_subset[fixed_subset.apply(
        lambda row: (int(row['layer']), int(row['unit'])) in neurons_in_group, axis=1
    )]
    
    for method, subset in [('silhouette', sil_subset), ('fixed-k5', fixed_subset)]:
        if len(subset) > 0:
            max_aucs = subset.groupby(['layer', 'unit'])['AUC'].max()
            results.append({
                'k_group': k_group,
                'method': method,
                'n_neurons': len(max_aucs),
                'n_descriptions': len(subset),
                'mean_auc': subset['AUC'].mean(),
                'mean_of_max_auc': max_aucs.mean(),
                'std_auc': subset['AUC'].std(),
                'median_auc': subset['AUC'].median()
            })

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Visualization: Performance by k-group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_mean = results_df.pivot(index='k_group', columns='method', values='mean_auc')
pivot_mean.plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_ylabel('Mean AUC')
axes[0].set_xlabel('K-Group')
axes[0].set_title('Mean AUC by K-Group and Method')
axes[0].legend(title='Method')
axes[0].grid(axis='y', alpha=0.3)

pivot_max = results_df.pivot(index='k_group', columns='method', values='mean_of_max_auc')
pivot_max.plot(kind='bar', ax=axes[1], rot=0)
axes[1].set_ylabel('Mean of Max AUC per Neuron')
axes[1].set_xlabel('K-Group')
axes[1].set_title('Best Description per Neuron by K-Group')
axes[1].legend(title='Method')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Key insight
print('\n' + '='*70)
print('KEY INSIGHT: Does fixed-k5 struggle with low-k neurons?')
print('='*70)

low_k_fixed = results_df[(results_df['k_group'] == 'low_k (k≤5)') & (results_df['method'] == 'fixed-k5')]
low_k_sil = results_df[(results_df['k_group'] == 'low_k (k≤5)') & (results_df['method'] == 'silhouette')]
high_k_fixed = results_df[(results_df['k_group'] == 'high_k (k>5)') & (results_df['method'] == 'fixed-k5')]

if len(low_k_fixed) > 0 and len(high_k_fixed) > 0:
    perf_drop = low_k_fixed['mean_of_max_auc'].values[0] - high_k_fixed['mean_of_max_auc'].values[0]
    print(f'\nFixed-k5 on low-k neurons: {low_k_fixed["mean_of_max_auc"].values[0]:.4f}')
    print(f'Fixed-k5 on high-k neurons: {high_k_fixed["mean_of_max_auc"].values[0]:.4f}')
    print(f'Performance drop: {perf_drop:.4f} ({perf_drop/high_k_fixed["mean_of_max_auc"].values[0]*100:.1f}%)')
    
    if len(low_k_sil) > 0:
        gap = low_k_fixed['mean_of_max_auc'].values[0] - low_k_sil['mean_of_max_auc'].values[0]
        print(f'\nFixed-k5 vs Silhouette on low-k neurons: {"+" if gap > 0 else ""}{gap:.4f}')
        print(f'→ Fixed-k5 {"OUTPERFORMS" if gap > 0 else "underperforms"} silhouette on low-k neurons!')

### Silhouette Score Distributions from Log Files

In [ ]:
# Extract silhouette scores from log files
log_dir = '../logs'
scores_df_data = []

for fname in os.listdir(log_dir):
    if 'silhouette' in fname and fname.endswith('.log'):
        match = re.search(r'layer-(\d+)_unit-(\d+)', fname)
        if not match:
            continue
        
        layer = int(match.group(1))
        unit = int(match.group(2))
        k_val = neuron_k.get((layer, unit), None)
        
        with open(os.path.join(log_dir, fname), 'r') as f:
            content = f.read()
            match = re.search(r'K-selection scores: ({[^}]+})', content)
            if match:
                try:
                    scores_dict = eval(match.group(1))
                    for k, score in scores_dict.items():
                        scores_df_data.append({
                            'layer': layer,
                            'unit': unit,
                            'k_tried': k,
                            'silhouette_score': score,
                            'k_selected': k_val,
                            'k_group': 'low_k (k≤5)' if k_val and k_val <= 5 else 'high_k (k>5)'
                        })
                except:
                    pass

scores_df = pd.DataFrame(scores_df_data)
print(f'Extracted silhouette scores for {len(scores_df["unit"].unique())} neurons')
print(f'Total score entries: {len(scores_df)}')

In [ ]:
# Plot: Overall distribution with KDE
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for k in sorted(scores_df['k_tried'].unique()):
    subset = scores_df[scores_df['k_tried'] == k]['silhouette_score']
    axes[0].hist(subset, bins=20, alpha=0.3, label=f'k={k}')

axes[0].set_xlabel('Silhouette Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Silhouette Scores by K-Value (All Neurons)')
axes[0].legend(ncol=3, fontsize=9)
axes[0].grid(alpha=0.3)

scores_df.boxplot(column='silhouette_score', by='k_tried', ax=axes[1])
axes[1].set_xlabel('K Value Tried')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score Distribution by K-Value')
axes[1].get_figure().suptitle('')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot: Split by k-group (bimodal analysis)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, k_group in enumerate(['low_k (k≤5)', 'high_k (k>5)']):
    group_data = scores_df[scores_df['k_group'] == k_group]
    
    # Histogram
    for k in sorted(group_data['k_tried'].unique()):
        subset = group_data[group_data['k_tried'] == k]['silhouette_score']
        axes[idx, 0].hist(subset, bins=15, alpha=0.4, label=f'k={k}')
    
    axes[idx, 0].set_xlabel('Silhouette Score')
    axes[idx, 0].set_ylabel('Frequency')
    axes[idx, 0].set_title(f'{k_group}: Score Distribution')
    axes[idx, 0].legend(ncol=3, fontsize=8)
    axes[idx, 0].grid(alpha=0.3)
    
    # Mean score by k-value
    mean_scores = group_data.groupby('k_tried')['silhouette_score'].agg(['mean', 'std']).reset_index()
    axes[idx, 1].errorbar(mean_scores['k_tried'], mean_scores['mean'], 
                          yerr=mean_scores['std'], marker='o', capsize=5, linewidth=2)
    axes[idx, 1].set_xlabel('K Value')
    axes[idx, 1].set_ylabel('Mean Silhouette Score')
    axes[idx, 1].set_title(f'{k_group}: Mean Score by K')
    axes[idx, 1].grid(alpha=0.3)
    axes[idx, 1].set_xticks(sorted(group_data['k_tried'].unique()))

plt.tight_layout()
plt.show()

## Simpson's Paradox Investigation

The previous results showed something confusing: fixed-k5 appears better in BOTH k-groups individually, but silhouette has higher overall mean AUC. This is a classic case of Simpson's Paradox.

In [ ]:
# Investigate Simpson's Paradox
print("=" * 80)
print("SIMPSON'S PARADOX INVESTIGATION")
print("=" * 80)

# Show overall means
print('\n1. OVERALL (all neurons combined):')
for method in ['fixed-k5', 'silhouette']:
    subset = df[df['method'] == method]
    print(f'   {method:12}: mean AUC = {subset["AUC"].mean():.4f} (n={len(subset)} descriptions)')

# Show by k-group
print('\n2. BY K-GROUP:')
for k_group in ['low_k (k≤5)', 'high_k (k>5)']:
    print(f'\n   {k_group}:')
    
    # Get silhouette neurons in this group
    sil_subset = df[(df['method'] == 'silhouette') & (df['k_group'] == k_group)]
    neurons_in_group = set(sil_subset.groupby(['layer', 'unit']).size().index)
    
    # Get fixed-k5 for SAME neurons  
    fixed_subset = df[df['method'] == 'fixed-k5']
    fixed_subset = fixed_subset[fixed_subset.apply(
        lambda row: (int(row['layer']), int(row['unit'])) in neurons_in_group, axis=1
    )]
    
    for method, subset in [('silhouette', sil_subset), ('fixed-k5', fixed_subset)]:
        if len(subset) > 0:
            print(f'     {method:12}: mean AUC = {subset["AUC"].mean():.4f} (n={len(subset)} descriptions)')

# Explanation
sil_low = df[(df['method'] == 'silhouette') & (df['k_group'] == 'low_k (k≤5)')]
sil_high = df[(df['method'] == 'silhouette') & (df['k_group'] == 'high_k (k>5)')]
neurons_low = len(set(sil_low.groupby(['layer', 'unit']).size().index))
neurons_high = len(set(sil_high.groupby(['layer', 'unit']).size().index))

print('\n3. EXPLANATION:')
print(f'   Silhouette method:')
print(f'     - Low-k neurons ({neurons_low}): {len(sil_low)} descriptions (avg {len(sil_low)/neurons_low:.1f} per neuron)')
print(f'     - High-k neurons ({neurons_high}): {len(sil_high)} descriptions (avg {len(sil_high)/neurons_high:.1f} per neuron)')
print(f'\n   Fixed-k5 generates equal descriptions per neuron (5 each), so no weighting effect.')
print(f'   Silhouette generates MORE descriptions for high-k neurons ({len(sil_high)} vs {len(sil_low)}).')
print(f'   High-k neurons perform BETTER, so this weighted average pulls silhouette\'s overall mean UP.')

## Improved Silhouette Score Visualization

Plot silhouette scores with k-value on x-axis to better understand the distribution.

In [ ]:
# Create improved visualization: k on x-axis, silhouette score on y-axis
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Overall scatter plot with jitter
ax = axes[0, 0]
for layer in sorted(scores_df['layer'].unique()):
    layer_data = scores_df[scores_df['layer'] == layer]
    # Add small jitter to k_tried for visibility
    k_jittered = layer_data['k_tried'] + np.random.normal(0, 0.1, len(layer_data))
    ax.scatter(k_jittered, layer_data['silhouette_score'], alpha=0.3, s=20, label=f'Layer {layer}')

# Add mean line
mean_scores = scores_df.groupby('k_tried')['silhouette_score'].mean()
ax.plot(mean_scores.index, mean_scores.values, 'k-', linewidth=3, label='Mean', zorder=10)
ax.set_xlabel('K-value', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Overall: Silhouette Scores by K-value', fontsize=14, fontweight='bold')
ax.set_xticks(range(2, 11))
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Mean scores with error bars
ax = axes[0, 1]
mean_scores = scores_df.groupby('k_tried')['silhouette_score'].mean()
std_scores = scores_df.groupby('k_tried')['silhouette_score'].std()
ax.errorbar(mean_scores.index, mean_scores.values, yerr=std_scores.values, 
            fmt='o-', linewidth=2, markersize=8, capsize=5, capthick=2)
ax.set_xlabel('K-value', fontsize=12)
ax.set_ylabel('Mean Silhouette Score', fontsize=12)
ax.set_title('Mean Silhouette Score by K-value (with std dev)', fontsize=14, fontweight='bold')
ax.set_xticks(range(2, 11))
ax.grid(True, alpha=0.3)

# 3. Split by k-group: low_k vs high_k
ax = axes[1, 0]
for k_group, color in [('low_k (k≤5)', 'red'), ('high_k (k>5)', 'blue')]:
    group_data = scores_df[scores_df['k_group'] == k_group]
    mean_scores = group_data.groupby('k_tried')['silhouette_score'].mean()
    std_scores = group_data.groupby('k_tried')['silhouette_score'].std()
    ax.errorbar(mean_scores.index, mean_scores.values, yerr=std_scores.values,
               fmt='o-', linewidth=2, markersize=8, capsize=5, capthick=2,
               label=k_group, color=color, alpha=0.7)

ax.set_xlabel('K-value', fontsize=12)
ax.set_ylabel('Mean Silhouette Score', fontsize=12)
ax.set_title('Silhouette Scores by K-value (split by k-group)', fontsize=14, fontweight='bold')
ax.set_xticks(range(2, 11))
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Individual neuron trajectories
ax = axes[1, 1]
# Plot individual neurons as faint lines
for (layer, unit), neuron_data in scores_df.groupby(['layer', 'unit']):
    neuron_data_sorted = neuron_data.sort_values('k_tried')
    k_selected = neuron_data_sorted['k_selected'].iloc[0]
    color = 'red' if k_selected <= 5 else 'blue'
    ax.plot(neuron_data_sorted['k_tried'], neuron_data_sorted['silhouette_score'], 
           alpha=0.15, linewidth=1, color=color)

# Highlight mean trajectory
mean_scores = scores_df.groupby('k_tried')['silhouette_score'].mean()
ax.plot(mean_scores.index, mean_scores.values, 'k-', linewidth=3, label='Mean', zorder=10)

ax.set_xlabel('K-value', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Individual Neuron Trajectories (red=low_k, blue=high_k)', fontsize=14, fontweight='bold')
ax.set_xticks(range(2, 11))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/scores_df_by_k.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nSaved plot to: results/scores_df_by_k.png')

In [ ]:
# Additional statistics about k-selection
print('\nK-SELECTION STATISTICS:')
print('\nMean silhouette score by k-value:')
mean_by_k = scores_df.groupby('k_tried')['silhouette_score'].agg(['mean', 'std', 'count'])
print(mean_by_k)

print('\nK-values selected by silhouette method:')
k_selected_counts = scores_df.groupby('k_selected').size().div(len(scores_df.groupby(['layer', 'unit']))).mul(100)
for k, pct in k_selected_counts.items():
    print(f'  k={k}: {pct:.1f}% of neurons')

In [ ]:
# Visualize k-values selected by silhouette method
from collections import Counter

# Extract k-values for silhouette
silhouette_k_vals = []
for (layer, unit), k in neuron_k.items():
    # Check if this neuron used silhouette (by checking if description file exists)
    fname_pattern = f'gpt2-xl_layer-{layer}_unit-{unit}_silhouette'
    for fname in os.listdir(desc_dir):
        if fname_pattern in fname:
            silhouette_k_vals.append(k)
            break

# Count occurrences
k_counts = Counter(silhouette_k_vals)
total_neurons = len(silhouette_k_vals)

# Create bar plot
fig, ax = plt.subplots(figsize=(12, 6))

k_values = sorted(k_counts.keys())
counts = [k_counts[k] for k in k_values]
percentages = [count / total_neurons * 100 for count in counts]

# Create bars
bars = ax.bar(k_values, counts, color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)

# Highlight k=2 and k=10 (extremes)
for i, k in enumerate(k_values):
    if k == 2:
        bars[i].set_color('red')
        bars[i].set_alpha(0.8)
    elif k == 10:
        bars[i].set_color('darkgreen')
        bars[i].set_alpha(0.8)

# Add percentage labels on top of bars
for i, (k, count, pct) in enumerate(zip(k_values, counts, percentages)):
    ax.text(k, count + 0.5, f'{count}\n({pct:.1f}%)', 
           ha='center', va='bottom', fontsize=10, fontweight='bold')

# Styling
ax.set_xlabel('K-value Selected', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Neurons', fontsize=14, fontweight='bold')
ax.set_title('Silhouette Method: Distribution of Selected K-values\n(Red = k=2, Green = k=10)', 
           fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(k_values)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, max(counts) + 5)

# Add summary statistics as text box
stats_text = (
    f'Total neurons: {total_neurons}\n'
    f'Mean k: {np.mean(silhouette_k_vals):.2f}\n'
    f'Median k: {np.median(silhouette_k_vals):.0f}\n'
    f'Mode k: {max(k_counts, key=k_counts.get)}\n'
    f'k=2: {k_counts.get(2, 0)} ({k_counts.get(2, 0)/total_neurons*100:.1f}%)\n'
    f'k=10: {k_counts.get(10, 0)} ({k_counts.get(10, 0)/total_neurons*100:.1f}%)'
)
ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
       verticalalignment='top', horizontalalignment='right',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
       fontsize=11, fontfamily='monospace')

plt.tight_layout()
plt.savefig('../results/silhouette_k_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved plot to: results/silhouette_k_distribution.png')
print('\nBIMODAL DISTRIBUTION CONFIRMED:')
print(f'  Low-k (k≤5): {sum(count for k, count in k_counts.items() if k <= 5)} neurons ({sum(count for k, count in k_counts.items() if k <= 5)/total_neurons*100:.1f}%)')
print(f'  High-k (k>5): {sum(count for k, count in k_counts.items() if k > 5)} neurons ({sum(count for k, count in k_counts.items() if k > 5)/total_neurons*100:.1f}%)')

## BIC and Davies-Bouldin Score Distributions

Analyzing score distributions for all three k-selection methods to understand their biases.

In [ ]:
# Extract BIC and Davies-Bouldin scores from log files
all_method_scores = []

for method_name in ['silhouette', 'bic', 'davies_bouldin']:
    for fname in os.listdir(log_dir):
        if method_name.replace('_', '-') in fname and fname.endswith('.log'):
            match = re.search(r'layer-(\d+)_unit-(\d+)', fname)
            if not match:
                continue
            
            layer = int(match.group(1))
            unit = int(match.group(2))
            
            with open(os.path.join(log_dir, fname), 'r') as f:
                content = f.read()
                match = re.search(r'K-selection scores: ({[^}]+})', content)
                if match:
                    try:
                        scores_dict = eval(match.group(1))
                        for k, score in scores_dict.items():
                            all_method_scores.append({
                                'layer': layer,
                                'unit': unit,
                                'method': method_name,
                                'k_tried': k,
                                'score': score
                            })
                    except:
                        pass

all_scores_df = pd.DataFrame(all_method_scores)
print(f'Extracted scores for {len(all_scores_df)} entries across {len(all_scores_df["unit"].unique())} neurons')
print(f'\nMethods: {all_scores_df["method"].unique()}')

In [ ]:
# Plot distributions for all three methods
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

methods = ['silhouette', 'bic', 'davies_bouldin']
method_labels = ['Silhouette Score', 'BIC', 'Davies-Bouldin Index']

for idx, (method, label) in enumerate(zip(methods, method_labels)):
    method_data = all_scores_df[all_scores_df['method'] == method]
    
    # Top row: Distribution by k-value
    ax = axes[0, idx]
    for k in sorted(method_data['k_tried'].unique()):
        subset = method_data[method_data['k_tried'] == k]['score']
        ax.hist(subset, bins=20, alpha=0.3, label=f'k={k}')
    
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{method.upper()}: Score Distribution', fontsize=12, fontweight='bold')
    ax.legend(ncol=3, fontsize=8)
    ax.grid(alpha=0.3)
    
    # Bottom row: Mean score by k-value
    ax = axes[1, idx]
    mean_scores = method_data.groupby('k_tried')['score'].mean()
    std_scores = method_data.groupby('k_tried')['score'].std()
    
    ax.errorbar(mean_scores.index, mean_scores.values, yerr=std_scores.values,
               fmt='o-', linewidth=2, markersize=8, capsize=5, capthick=2)
    ax.set_xlabel('K-value', fontsize=11)
    ax.set_ylabel(f'Mean {label}', fontsize=11)
    ax.set_title(f'{method.upper()}: Mean Score by K', fontsize=12, fontweight='bold')
    ax.set_xticks(range(2, 11))
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/all_methods_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved plot to: results/all_methods_distributions.png')

In [ ]:
# Summary statistics for each method
print('\n' + '='*80)
print('SCORE STATISTICS BY METHOD')
print('='*80)

for method in methods:
    method_data = all_scores_df[all_scores_df['method'] == method]
    
    print(f'\n{method.upper()}:')
    print(f'  Overall mean: {method_data["score"].mean():.4f}')
    print(f'  Overall std:  {method_data["score"].std():.4f}')
    print(f'  Overall CV:   {method_data["score"].std() / abs(method_data["score"].mean()):.4f}')
    
    # Score variation within each neuron
    neuron_ranges = []
    for (layer, unit), neuron_data in method_data.groupby(['layer', 'unit']):
        score_range = neuron_data['score'].max() - neuron_data['score'].min()
        neuron_ranges.append(score_range)
    
    print(f'  Mean variation per neuron: {np.mean(neuron_ranges):.4f} ± {np.std(neuron_ranges):.4f}')

## K-Selection Bias Analysis

Examining which k-values each method selected and their inherent biases.

In [ ]:
# Extract selected k-values for each method
k_selections = {}

for method_name in ['silhouette', 'bic', 'davies_bouldin']:
    k_selections[method_name] = []
    
    for fname in os.listdir(desc_dir):
        if method_name.replace('_', '-') in fname and fname.endswith('.csv'):
            df_desc = pd.read_csv(os.path.join(desc_dir, fname))
            k_selected = len(df_desc)
            k_selections[method_name].append(k_selected)

print('K-SELECTION RESULTS:')
print('='*80)

for method_name in ['silhouette', 'bic', 'davies_bouldin']:
    k_vals = k_selections[method_name]
    print(f'\n{method_name.upper()}:')
    
    from collections import Counter
    k_counts = Counter(k_vals)
    
    for k in sorted(k_counts.keys()):
        count = k_counts[k]
        pct = count / len(k_vals) * 100
        bar = '█' * int(pct / 2)
        print(f'  k={k:2d}: {count:2d} neurons ({pct:5.1f}%) {bar}')
    
    print(f'  Mean k: {np.mean(k_vals):.2f}')
    print(f'  Median k: {np.median(k_vals):.1f}')

In [ ]:
# Visualize k-selection distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (method_name, label) in enumerate(zip(
    ['silhouette', 'bic', 'davies_bouldin'],
    ['Silhouette', 'BIC', 'Davies-Bouldin']
)):
    k_vals = k_selections[method_name]
    
    # Create histogram
    ax = axes[idx]
    counts = {k: k_vals.count(k) for k in range(2, 11)}
    
    bars = ax.bar(counts.keys(), counts.values(), color='steelblue', alpha=0.7, edgecolor='black')
    
    # Highlight most common k
    most_common_k = max(counts, key=counts.get)
    bars[most_common_k - 2].set_color('red')
    bars[most_common_k - 2].set_alpha(0.8)
    
    ax.set_xlabel('K-value Selected', fontsize=12)
    ax.set_ylabel('Number of Neurons', fontsize=12)
    ax.set_title(f'{label}\n(Most common: k={most_common_k})', fontsize=13, fontweight='bold')
    ax.set_xticks(range(2, 11))
    ax.grid(axis='y', alpha=0.3)
    
    # Add percentage labels
    for k, count in counts.items():
        if count > 0:
            pct = count / len(k_vals) * 100
            ax.text(k, count + 1, f'{pct:.0f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/k_selection_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved plot to: results/k_selection_comparison.png')

## Example Neurons: Low-k vs High-k

Detailed examination of two neurons to understand why one selected k=2 and another selected k=10.
We'll look at all three metrics (Silhouette, BIC, Davies-Bouldin) for both neurons.

In [ ]:
# Find representative neurons
# Low-k neuron: Layer 40, Unit 183 (selected k=2)
# High-k neuron: Layer 40, Unit 4055 (selected k=10)

example_neurons = [
    {'layer': 40, 'unit': 183, 'label': 'Low-k Example (k=2)'},
    {'layer': 40, 'unit': 4055, 'label': 'High-k Example (k=10)'}
]

# Extract scores for these neurons across all methods
example_scores = {}

for neuron in example_neurons:
    layer, unit, label = neuron['layer'], neuron['unit'], neuron['label']
    example_scores[label] = {}
    
    for method in ['silhouette', 'bic', 'davies_bouldin']:
        # Find log file
        pattern = f'gpt2-xl_layer-{layer}_unit-{unit}_{method.replace("_", "-")}'
        
        for fname in os.listdir(log_dir):
            if pattern in fname and fname.endswith('.log'):
                with open(os.path.join(log_dir, fname), 'r') as f:
                    content = f.read()
                    match = re.search(r'K-selection scores: ({[^}]+})', content)
                    if match:
                        try:
                            scores_dict = eval(match.group(1))
                            example_scores[label][method] = scores_dict
                        except:
                            pass
                break

# Verify data extraction
for label, methods_data in example_scores.items():
    print(f'{label}:')
    for method in methods_data.keys():
        print(f'  {method}: {len(methods_data[method])} k-values')

In [ ]:
# Create comparison table
print('='*100)
print('DETAILED SCORE COMPARISON: LOW-K vs HIGH-K NEURONS')
print('='*100)

for label in example_scores.keys():
    print(f'\n{label}')
    print('-'*100)
    
    # Create table
    k_values = range(2, 11)
    print(f'{"k":>3} | {"Silhouette":>12} | {"BIC":>12} | {"Davies-Bouldin":>15}')
    print('-'*100)
    
    for k in k_values:
        sil_score = example_scores[label].get('silhouette', {}).get(k, 'N/A')
        bic_score = example_scores[label].get('bic', {}).get(k, 'N/A')
        db_score = example_scores[label].get('davies_bouldin', {}).get(k, 'N/A')
        
        # Format scores
        sil_str = f'{sil_score:.6f}' if isinstance(sil_score, (int, float)) else sil_score
        bic_str = f'{bic_score:.2f}' if isinstance(bic_score, (int, float)) else bic_score
        db_str = f'{db_score:.4f}' if isinstance(db_score, (int, float)) else db_score
        
        # Mark winners
        sil_mark = ' ← MAX' if isinstance(sil_score, (int, float)) and sil_score == max([s for s in example_scores[label].get('silhouette', {}).values() if isinstance(s, (int, float))], default=0) else ''
        bic_mark = ' ← MIN' if isinstance(bic_score, (int, float)) and bic_score == min([s for s in example_scores[label].get('bic', {}).values() if isinstance(s, (int, float))], default=float('inf')) else ''
        db_mark = ' ← MIN' if isinstance(db_score, (int, float)) and db_score == min([s for s in example_scores[label].get('davies_bouldin', {}).values() if isinstance(s, (int, float))], default=float('inf')) else ''
        
        print(f'{k:3d} | {sil_str:>12}{sil_mark:6} | {bic_str:>12}{bic_mark:6} | {db_str:>15}{db_mark:6}')
    
    # Print statistics
    print('\nStatistics:')
    for method in ['silhouette', 'bic', 'davies_bouldin']:
        if method in example_scores[label]:
            scores = list(example_scores[label][method].values())
            score_range = max(scores) - min(scores)
            print(f'  {method:20}: range = {score_range:.6f}, mean = {np.mean(scores):.6f}, std = {np.std(scores):.6f}')

In [ ]:
# Visualize score trajectories for both neurons
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

methods = ['silhouette', 'bic', 'davies_bouldin']
method_labels = ['Silhouette Score\n(higher = better)', 'BIC\n(lower = better)', 'Davies-Bouldin\n(lower = better)']

for neuron_idx, (label, neuron_data) in enumerate(example_scores.items()):
    for method_idx, (method, method_label) in enumerate(zip(methods, method_labels)):
        ax = axes[neuron_idx, method_idx]
        
        if method in neuron_data:
            scores_dict = neuron_data[method]
            k_vals = sorted(scores_dict.keys())
            scores = [scores_dict[k] for k in k_vals]
            
            # Plot trajectory
            ax.plot(k_vals, scores, 'o-', linewidth=2, markersize=8, color='steelblue')
            
            # Mark selected k
            if method == 'silhouette':
                selected_k = max(scores_dict, key=scores_dict.get)
            else:  # bic or davies_bouldin
                selected_k = min(scores_dict, key=scores_dict.get)
            
            selected_score = scores_dict[selected_k]
            ax.plot(selected_k, selected_score, 'r*', markersize=20, label=f'Selected: k={selected_k}')
            
            # Add score range annotation
            score_range = max(scores) - min(scores)
            ax.text(0.05, 0.95, f'Range: {score_range:.6f}', 
                   transform=ax.transAxes, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
            
            ax.set_xlabel('K-value', fontsize=11)
            ax.set_ylabel(method_label, fontsize=11)
            ax.set_xticks(range(2, 11))
            ax.grid(alpha=0.3)
            ax.legend()
        
        # Add title to first column
        if method_idx == 0:
            ax.set_title(f'{label}\n{method.upper()}', fontsize=12, fontweight='bold')
        else:
            ax.set_title(method.upper(), fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/example_neurons_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved plot to: results/example_neurons_comparison.png')

In [ ]:
# Key insights
print('\n' + '='*100)
print('KEY INSIGHTS FROM EXAMPLE NEURONS')
print('='*100)

print('\n1. SCORE VARIATION:')
print('   When scores vary by tiny amounts (e.g., 0.001-0.003 for silhouette), k-selection')
print('   is essentially RANDOM. The "winner" is determined by noise, not meaningful structure.')

print('\n2. LOW-K NEURONS (e.g., k=2):')
print('   - All silhouette scores are similarly low (~0.014-0.017)')
print('   - Tiny differences determine winner')
print('   - k=2 wins by random variation, NOT because neuron is monosemantic')

print('\n3. HIGH-K NEURONS (e.g., k=10):')
print('   - Also have low silhouette scores (~0.012-0.016)')
print('   - NO BETTER clustering quality than low-k neurons')
print('   - k=10 wins by random variation')

print('\n4. EXPECTED WITH COSINE SIMILARITY:')
print('   - Silhouette scores should be 0.1-0.4 (10-25x higher!)')
print('   - Score differences should be meaningful (>0.05)')
print('   - Clear winner emerges, not random noise')

print('\n5. DAVIES-BOULDIN BENCHMARKS:')
for label, neuron_data in example_scores.items():
    if 'davies_bouldin' in neuron_data:
        db_scores = list(neuron_data['davies_bouldin'].values())
        mean_db = np.mean(db_scores)
        print(f'   {label}: mean DB = {mean_db:.2f} (threshold: <4.0 = acceptable, yours: {"POOR" if mean_db > 4.0 else "OK"})')

In [ ]:
# Visualize BIC behavior: Why does it always choose k=2?
print('='*80)
print('BIC STATISTICS EXPLAINED')
print('='*80)

bic_data = all_scores_df[all_scores_df['method'] == 'bic']

print('\n1. OVERALL STATISTICS:')
print(f'   Mean BIC: {bic_data["score"].mean():.2f}')
print(f'   Std BIC:  {bic_data["score"].std():.2f}')
print(f'   CV:       {bic_data["score"].std() / bic_data["score"].mean():.4f}')

print('\n2. VARIATION WITHIN NEURONS:')
neuron_ranges = []
neuron_means = []
for (layer, unit), neuron_data in bic_data.groupby(['layer', 'unit']):
    score_range = neuron_data['score'].max() - neuron_data['score'].min()
    neuron_ranges.append(score_range)
    neuron_means.append(neuron_data['score'].mean())

mean_range = np.mean(neuron_ranges)
mean_bic = np.mean(neuron_means)

print(f'   Mean variation per neuron: {mean_range:.2f}')
print(f'   Mean BIC per neuron: {mean_bic:.2f}')
print(f'   Variation as % of mean: {mean_range / mean_bic * 100:.1f}%')

print('\n3. BIC TREND WITH K:')
mean_by_k = bic_data.groupby('k_tried')['score'].mean()
print('   K-value | Mean BIC   | Change from k=2')
print('   --------|------------|------------------')
k2_mean = mean_by_k[2]
for k in sorted(mean_by_k.index):
    change = mean_by_k[k] - k2_mean
    print(f'   {k:7d} | {mean_by_k[k]:10.2f} | {change:+10.2f} ({change/k2_mean*100:+.1f}%)')

print('\n4. INTERPRETATION:')
if mean_by_k[10] > mean_by_k[2]:
    print('   ✗ BIC INCREASES with k → Complexity penalty dominates')
    print('   ✗ Adding clusters makes model WORSE, not better')
    print('   ✗ BIC will always choose k=2 (simplest model)')
else:
    print('   ✓ BIC decreases with k → Clustering is working')

# Visualize BIC trend
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: BIC trend by k-value
ax = axes[0]
mean_by_k = bic_data.groupby('k_tried')['score'].mean()
std_by_k = bic_data.groupby('k_tried')['score'].std()

ax.plot(mean_by_k.index, mean_by_k.values, 'o-', linewidth=3, markersize=10, color='steelblue')
ax.fill_between(mean_by_k.index, 
                mean_by_k.values - std_by_k.values,
                mean_by_k.values + std_by_k.values,
                alpha=0.3, color='steelblue')

# Mark k=2 (always selected)
ax.plot(2, mean_by_k[2], 'r*', markersize=25, label='Always selected (k=2)')

ax.set_xlabel('K-value', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean BIC Score', fontsize=13, fontweight='bold')
ax.set_title('BIC Trend: Why Always k=2?\n(Lower is better)', fontsize=14, fontweight='bold')
ax.set_xticks(range(2, 11))
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Add annotation
if mean_by_k[10] > mean_by_k[2]:
    ax.annotate('BIC increases with k\n→ Complexity penalty dominates',
               xy=(6, mean_by_k[6]), xytext=(7, mean_by_k[2] + 5000),
               arrowprops=dict(arrowstyle='->', color='red', lw=2),
               fontsize=11, color='red', fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# Right: Variation within neurons
ax = axes[1]
ax.hist(neuron_ranges, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(mean_range, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_range:.0f}')
ax.set_xlabel('BIC Range (max - min) per Neuron', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Neurons', fontsize=13, fontweight='bold')
ax.set_title('BIC Variation Within Neurons\n(How much does BIC change across k=2 to k=10?)', 
           fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/bic_behavior_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nSaved plot to: results/bic_behavior_analysis.png')